# 0 — Build Chunk Registry

Creates `data/chunk_registry_v1.parquet` — the master index of all annotated video chunks,
enriched with study-level metadata (conference, session outcomes, temporal position).

**Run this before Stage 3** (`3a-sample_validation_set.ipynb`), which adds the
`human_validation_set` flag to produce `chunk_registry_v2.parquet`.

**Inputs:**
- `../../gemini_code/<conferenceID>_path_dict.json` — one per conference
- `../../gemini_code/analysis_v1/data/<conferenceID>/<conferenceID>_session_outcomes.json` — one per conference

**Output:**
- `../data/chunk_registry_v1.parquet`
- `../data/chunk_registry_v1.csv` (human-readable backup)

## Step 0 — Imports and paths

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

# ── Resolve directories relative to this notebook ────────────────────────────
NOTEBOOK_DIR   = Path().resolve()                        # analysis_v2/notebooks/
ANALYSIS_V2    = NOTEBOOK_DIR.parent                     # analysis_v2/
GEMINI_CODE    = ANALYSIS_V2.parent.parent / 'gemini_code'  # NICO/gemini_code/
ANALYSIS_V1    = GEMINI_CODE / 'analysis_v1' / 'data'   # analysis_v1/data/
OUTPUT_DIR     = ANALYSIS_V2 / 'data'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('GEMINI_CODE :', GEMINI_CODE)
print('ANALYSIS_V1 :', ANALYSIS_V1)
print('OUTPUT_DIR  :', OUTPUT_DIR)

GEMINI_CODE : /Users/eveyhuang/Documents/NICO/gemini_code
ANALYSIS_V1 : /Users/eveyhuang/Documents/NICO/gemini_code/analysis_v1/data
OUTPUT_DIR  : /Users/eveyhuang/Documents/NICO/gemini_code/analysis_v2/data


## Step 1 — Discover inputs

In [2]:
# Each path_dict file name encodes the conference ID: e.g. '2020NES_path_dict.json' → '2020NES'
path_dict_files = sorted(GEMINI_CODE.glob('*_path_dict.json'))

print(f'Found {len(path_dict_files)} path_dict files:')
for f in path_dict_files:
    conference_id = f.name.replace('_path_dict.json', '')
    outcomes_path = ANALYSIS_V1 / conference_id / f'{conference_id}_session_outcomes.json'
    status = '✓' if outcomes_path.exists() else '✗ MISSING'
    print(f'  {conference_id:12s}  path_dict={f.name}  outcomes={status}')

Found 8 path_dict files:
  2020NES       path_dict=2020NES_path_dict.json  outcomes=✓
  2021ABI       path_dict=2021ABI_path_dict.json  outcomes=✓
  2021CMC       path_dict=2021CMC_path_dict.json  outcomes=✓
  2021MND       path_dict=2021MND_path_dict.json  outcomes=✓
  2021MZT       path_dict=2021MZT_path_dict.json  outcomes=✓
  2021NES       path_dict=2021NES_path_dict.json  outcomes=✓
  2021SLU       path_dict=2021SLU_path_dict.json  outcomes=✓
  2022MND       path_dict=2022MND_path_dict.json  outcomes=✓


## Step 2 — Helper: parse session outcomes

Each `*_session_outcomes.json` maps `session_group_key → {teams: {team_id: {members, funded_status}}}`.

We flatten this to per-session-group outcome counts.

In [10]:
def normalize_key(s: str) -> str:
    """Normalize session key for matching: lowercase, replace - and _ with space."""
    return s.replace('-', ' ').replace('_', ' ').lower()

def parse_session_outcomes(outcomes_path: Path) -> dict:
    """
    Returns dict keyed by NORMALIZED session_group for matching with path_dict keys.
    Raw keys vary between files: some use underscores (2020_11_05_NES_S1),
    others use hyphens (2021-05-20-ABI-S2). normalize_key() makes both comparable.

    Note: some files contain extra top-level keys with non-dict values
    (e.g. 'missing_names': [], 'people_not_in_any_team': [...]).
    These are silently skipped.
    """
    with open(outcomes_path) as f:
        raw = json.load(f)

    result = {}
    for session_group, data in raw.items():
        if not isinstance(data, dict):
            continue   # skip metadata keys like 'missing_names', 'people_not_in_any_team'
        teams = data.get('teams', {})
        num_teams        = len(teams)
        num_funded_teams = sum(1 for t in teams.values() if t.get('funded_status', 0) == 1)
        result[normalize_key(session_group)] = {
            'num_teams':        num_teams,
            'num_funded_teams': num_funded_teams,
            'has_teams':        num_teams > 0,
            'has_funded_teams': num_funded_teams > 0,
        }
    return result

# Quick smoke-test on 2020NES
_test_path = ANALYSIS_V1 / '2020NES' / '2020NES_session_outcomes.json'
_test = parse_session_outcomes(_test_path)
print(f'2020NES: {len(_test)} session groups')
for k, v in list(_test.items())[:3]:
    print(f'  {k!r}: {v}')

2020NES: 16 session groups
  '2020 11 05 nes s1': {'num_teams': 1, 'num_funded_teams': 0, 'has_teams': True, 'has_funded_teams': False}
  '2020 11 06 nes s7': {'num_teams': 2, 'num_funded_teams': 0, 'has_teams': True, 'has_funded_teams': False}
  '2020 11 06 nes s10': {'num_teams': 0, 'num_funded_teams': 0, 'has_teams': False, 'has_funded_teams': False}


## Step 3 — Helper: temporal position label

Assigns `beginning` / `middle` / `end` based on chunk index within the session recording.

A session recording may have multiple recordings (sub-sessions), each with their own chunks.
Position is relative to chunks **within a single recording** (= one path_dict session_key).

In [7]:
def chunk_position_label(chunk_index: int, n_chunks: int) -> str:
    if n_chunks == 1:
        return 'whole'
    if chunk_index == 0:
        return 'beginning'
    if chunk_index == n_chunks - 1:
        return 'end'
    return 'middle'

## Step 4 — Build the registry

Expand every path_dict entry into one row per chunk, then join outcomes.

In [17]:
rows = []
unmatched_sessions = []   # session_groups with no outcome entry
missing_outcomes   = []   # conferences missing an outcomes file

for pd_file in path_dict_files:
    conference_id  = pd_file.name.replace('_path_dict.json', '')
    outcomes_path  = ANALYSIS_V1 / conference_id / f'{conference_id}_session_outcomes.json'

    if not outcomes_path.exists():
        missing_outcomes.append(conference_id)
        outcomes_lookup = {}
    else:
        outcomes_lookup = parse_session_outcomes(outcomes_path)

    with open(pd_file) as f:
        path_dict = json.load(f)

    for session_key, chunks in path_dict.items():
        # session_key = '<session_group>/<recording_folder>'
        # e.g. '2020_11_05_NES_S1/1_DAC_Simulations_Zoom_Meeting_2020_11_05_12_09_10'
        session_group = session_key.split('/')[0]   # e.g. '2020_11_05_NES_S1'

        # Look up outcomes for this session group (normalize to handle - vs _ differences)
        outcome = outcomes_lookup.get(normalize_key(session_group))
        if outcome is None:
            unmatched_sessions.append((conference_id, session_group))
            outcome = {
                'num_teams': None, 'num_funded_teams': None,
                'has_teams': None, 'has_funded_teams': None,
            }

        n_chunks = len(chunks)
        for i, chunk in enumerate(chunks):
            # path_dict chunk schema: [chunk_file_name, chunk_path, gemini_file_name, analyzed_flag]
            # (4 elements; model_used is not stored in current implementation)
            chunk_file_name = chunk[0]
            chunk_path      = chunk[1]
            gemini_file_ref = chunk[2]
            analyzed_flag   = bool(chunk[3])
            if len(chunk) > 4:
                if chunk[4]:
                    model_used = chunk[4]
                else:
                    model_used = 'gemini-3.1-pro-preview'
            else:
                model_used = 'gemini-3.1-pro-preview'

            rows.append({
                'chunk_id':              f'{conference_id}__{session_key}__{chunk_file_name}',
                'conference_id':         conference_id,
                'session_key':           session_key,
                'session_group':         session_group,
                'chunk_file_name':       chunk_file_name,
                'chunk_path':            chunk_path,
                'gemini_file_ref':       gemini_file_ref,
                'chunk_index':           i,
                'n_chunks_in_session':   n_chunks,
                'chunk_position':        chunk_position_label(i, n_chunks),
                'analyzed':              analyzed_flag,
                'model_used':            model_used,
                # Outcomes
                'num_teams':             outcome['num_teams'],
                'num_funded_teams':      outcome['num_funded_teams'],
                'has_teams':             outcome['has_teams'],
                'has_funded_teams':      outcome['has_funded_teams'],
                # Validation flags (populated in Stage 3a)
                'human_validation_set':     False,
                'utterance_validation_set': False,
                'oversampled_for':          None,
            })

registry = pd.DataFrame(rows)

# Convenience bool for stratification key (same as has_funded_teams but explicit)
registry['outcome_has_funded_teams'] = registry['has_funded_teams'].astype('boolean')

print(f'Registry: {len(registry):,} chunks from {registry["session_key"].nunique():,} recordings '
      f'across {registry["conference_id"].nunique()} conferences')
if missing_outcomes:
    print(f'\nWARNING: Missing outcomes files for: {missing_outcomes}')
if unmatched_sessions:
    print(f'\nWARNING: {len(unmatched_sessions)} session_groups had no outcome entry:')
    for conf, sg in unmatched_sessions:
        print(f'  [{conf}] {sg}')

Registry: 1,310 chunks from 196 recordings across 8 conferences

  [2020NES] 2020_11_06_NES_S6
  [2020NES] 2020_11_06_NES_S6
  [2020NES] 2020_11_06_NES_S6
  [2020NES] 2020_11_06_NES_S6
  [2021MZT] 2021_09_30_MZT_S1
  [2021MZT] 2021_10_01_MZT_S6
  [2021MZT] 2021_10_01_MZT_S10
  [2021MZT] 2021_10_01_MZT_S12
  [2021MZT] 2021_10_01_MZT_S13


## Step 5 — Validate

In [12]:
# ── Uniqueness ────────────────────────────────────────────────────────────────
assert registry['chunk_id'].is_unique, \
    f'chunk_id not unique! Duplicates: {registry[registry["chunk_id"].duplicated()]["chunk_id"].tolist()[:5]}'

# ── All chunks have a conference_id ──────────────────────────────────────────
assert registry['conference_id'].notna().all(), 'Some chunks are missing conference_id'

# ── Outcome coverage ─────────────────────────────────────────────────────────
n_missing_outcome = registry['has_funded_teams'].isna().sum()
if n_missing_outcome > 0:
    print(f'NOTE: {n_missing_outcome} chunks have no outcome data (NaN in has_funded_teams)')
else:
    print('✓ All chunks have outcome data')

print('\n── Distribution by conference ────────────────────────────────────────────')
print(registry.groupby('conference_id')['chunk_id'].count().rename('n_chunks').to_string())

print('\n── Distribution by conference × chunk_position ──────────────────────────')
print(
    registry.groupby(['conference_id', 'chunk_position'])['chunk_id']
    .count()
    .unstack(fill_value=0)
    .to_string()
)

print('\n── Analyzed flag ────────────────────────────────────────────────────────')
print(registry['analyzed'].value_counts().to_string())

print('\n── Outcome rates per conference ─────────────────────────────────────────')
# Deduplicate to session_group level before computing rates
session_level = registry.drop_duplicates('session_group')[[
    'conference_id', 'session_group', 'has_teams', 'has_funded_teams'
]]
print(
    session_level.groupby('conference_id')[['has_teams', 'has_funded_teams']]
    .mean()
    .round(2)
    .to_string()
)

NOTE: 50 chunks have no outcome data (NaN in has_funded_teams)

── Distribution by conference ────────────────────────────────────────────
conference_id
2020NES    142
2021ABI    203
2021CMC    167
2021MND    170
2021MZT    168
2021NES    163
2021SLU    167
2022MND    130

── Distribution by conference × chunk_position ──────────────────────────
chunk_position  beginning  end  middle  whole
conference_id                                
2020NES                26   26      65     25
2021ABI                24   24     155      0
2021CMC                21   21     125      0
2021MND                23   23     123      1
2021MZT                20   20     128      0
2021NES                18   18     127      0
2021SLU                20   20     127      0
2022MND                16   16      96      2

── Analyzed flag ────────────────────────────────────────────────────────
analyzed
True    1310

── Outcome rates per conference ─────────────────────────────────────────
              has_te

## Step 6 — Inspect sample rows

In [18]:
# Show one recording with multiple chunks to verify position labels
multi_chunk = registry[registry['n_chunks_in_session'] > 3]['session_key'].iloc[0]
print(f'Example multi-chunk recording: {multi_chunk}\n')
display(
    registry[registry['session_key'] == multi_chunk][
        ['chunk_file_name', 'chunk_index', 'n_chunks_in_session',
         'chunk_position', 'analyzed', 'has_funded_teams']
    ]
)

Example multi-chunk recording: 2020_11_05_NES_S5/5_Decarbonizing_the_Economy_Zoom_Meeting_11_5_2020_12_19_34_PM



,chunk_file_name,chunk_index,n_chunks_in_session,chunk_position,analyzed,has_funded_teams
4,5_Decarbonizing_the_Economy_Zoom_Meeting_11_5_...,0,6,beginning,True,True
5,5_Decarbonizing_the_Economy_Zoom_Meeting_11_5_...,1,6,middle,True,True
6,5_Decarbonizing_the_Economy_Zoom_Meeting_11_5_...,2,6,middle,True,True
7,5_Decarbonizing_the_Economy_Zoom_Meeting_11_5_...,3,6,middle,True,True
8,5_Decarbonizing_the_Economy_Zoom_Meeting_11_5_...,4,6,middle,True,True
9,5_Decarbonizing_the_Economy_Zoom_Meeting_11_5_...,5,6,end,True,True


## Step 7 — Save

In [19]:
out_parquet = OUTPUT_DIR / 'chunk_registry_v1.parquet'
out_csv     = OUTPUT_DIR / 'chunk_registry_v1.csv'


registry.to_csv(out_csv, index=False)

print(f'Saved {len(registry):,} rows → {out_parquet}')
print(f'Saved CSV backup       → {out_csv}')
print()
print('Columns:')
print(registry.dtypes.to_string())

Saved 1,310 rows → /Users/eveyhuang/Documents/NICO/gemini_code/analysis_v2/data/chunk_registry_v1.parquet
Saved CSV backup       → /Users/eveyhuang/Documents/NICO/gemini_code/analysis_v2/data/chunk_registry_v1.csv

Columns:
chunk_id                     object
conference_id                object
session_key                  object
session_group                object
chunk_file_name              object
chunk_path                   object
gemini_file_ref              object
chunk_index                   int64
n_chunks_in_session           int64
chunk_position               object
analyzed                       bool
model_used                   object
num_teams                   float64
num_funded_teams            float64
has_teams                    object
has_funded_teams             object
human_validation_set           bool
utterance_validation_set       bool
oversampled_for              object
outcome_has_funded_teams    boolean
